You have the marketing_campaign table, which records in-app purchases by users. Users making their first in-app purchase enter a marketing campaign, where they see call-to-actions for more purchases. Find how many users made additional purchases due to the campaign's success.


The campaign starts one day after the first purchase. Users with only one or multiple purchases on the first day do not count, nor do users who later buy only the same products from their first day.


**Goal**

Count users who:

Made a first purchase (day 1)
Then (from next day onwards) made at least one purchase
And that later purchase includes at least one product NOT bought on day 1


**Key Logic
**
Identify each user’s first purchase date
Collect products bought on first day
Look at purchases after first day
Check if user bought any new product
Count such users

In [0]:
WITH first_day AS (
    SELECT 
        user_id,
        MIN(purchase_date) AS first_date
    FROM marketing_campaign
    GROUP BY user_id
),

first_day_products AS (
    SELECT DISTINCT 
        m.user_id,
        m.product_id
    FROM marketing_campaign m
    JOIN first_day f 
        ON m.user_id = f.user_id
       AND m.purchase_date = f.first_date
),

later_purchases AS (
    SELECT 
        m.user_id,
        m.product_id
    FROM marketing_campaign m
    JOIN first_day f 
        ON m.user_id = f.user_id
    WHERE m.purchase_date > f.first_date
),

valid_users AS (
    SELECT DISTINCT lp.user_id
    FROM later_purchases lp
    LEFT JOIN first_day_products fp
        ON lp.user_id = fp.user_id
       AND lp.product_id = fp.product_id
    WHERE fp.product_id IS NULL   -- new product
)

SELECT COUNT(*) AS users_converted
FROM valid_users;

In [0]:
from pyspark.sql import functions as F

# Step 1: First purchase date
first_day = df.groupBy("user_id").agg(
    F.min("purchase_date").alias("first_date")
)

# Step 2: First day products
first_day_products = df.join(first_day, "user_id") \
    .filter(F.col("purchase_date") == F.col("first_date")) \
    .select("user_id", "product_id").distinct()

# Step 3: Later purchases
later_purchases = df.join(first_day, "user_id") \
    .filter(F.col("purchase_date") > F.col("first_date")) \
    .select("user_id", "product_id")

# Step 4: Find new product purchases
valid_users = later_purchases.join(
    first_day_products,
    ["user_id", "product_id"],
    "left_anti"   # products not in first day
).select("user_id").distinct()

# Step 5: Count
result = valid_users.count()
print(result)

Edge Cases Covered
Multiple purchases on first day → treated as baseline
No purchases after first day → excluded
Only repeat products → excluded
At least one new product later → included